In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully.')

Mounted at /content/drive
Drive mounted successfully.


In [2]:
# Imports
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import numpy as np

matplotlib.use('Agg')

print(f'matplotlib version: {matplotlib.__version__}')
print(f'pandas version: {pd.__version__}')

matplotlib version: 3.10.0
pandas version: 2.2.2


In [3]:
# Define paths
RESULTS_DIR = Path('/content/drive/MyDrive/HRC_Research/results/occlusion_benchmark')
FIGURES_DIR = Path('/content/drive/MyDrive/HRC_Research/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

OCCLUSION_CSV    = RESULTS_DIR / 'occlusion_results_n30_70_10_20.csv'
CONSENSUS_CSV    = RESULTS_DIR / 'consensus_benchmark_results_70_10_20.csv'

# Verify files exist before proceeding
for f in [OCCLUSION_CSV, CONSENSUS_CSV]:
    assert f.exists(), f'FILE NOT FOUND: {f}'
    print(f'Found: {f}')
print('All paths verified.')

Found: /content/drive/MyDrive/HRC_Research/results/occlusion_benchmark/occlusion_results_n30_70_10_20.csv
Found: /content/drive/MyDrive/HRC_Research/results/occlusion_benchmark/consensus_benchmark_results_70_10_20.csv
All paths verified.


In [4]:
# Load locked results from CSV
occ_df = pd.read_csv(OCCLUSION_CSV)
con_df = pd.read_csv(CONSENSUS_CSV)

# X axis values (occlusion rates as integers for plotting)
x_rates = [0, 10, 20, 30, 50]

# Read mean values directly from the CSV columns
stgcn_means  = occ_df['stgcn_mean'].values
ctrgcn_means = occ_df['ctrgcn_mean'].values
con_means    = con_df['con_mean'].values

print('ST-GCN means  :', [f'{v:.2f}' for v in stgcn_means])
print('CTR-GCN means :', [f'{v:.2f}' for v in ctrgcn_means])
print('Consensus means:', [f'{v:.2f}' for v in con_means])

locked_improvements = {
    '0%':  -0.17,
    '10%':  3.33,
    '20%':  3.08,
    '30%':  4.34,
    '50%':  0.95,
}
imp_values = list(locked_improvements.values())
print('Locked improvements:', imp_values)

ST-GCN means  : ['55.27', '43.55', '29.61', '23.48', '16.92']
CTR-GCN means : ['67.69', '50.90', '33.88', '23.61', '13.29']
Consensus means: ['67.52', '54.22', '36.96', '27.95', '17.86']
Locked improvements: [-0.17, 3.33, 3.08, 4.34, 0.95]


In [5]:
# IEEE publication style settings

IEEE_RC = {
    'font.family':        'serif',
    'font.serif':         ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':          10,
    'axes.labelsize':     10,
    'axes.titlesize':     10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    8,
    'figure.dpi':         300,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.02,
    'lines.linewidth':    1.2,
    'lines.markersize':   4,
}

plt.rcParams.update(IEEE_RC)

IEEE_W = 3.5
IEEE_H = 2.625

print('IEEE style applied.')
print(f'Figure size: {IEEE_W} x {IEEE_H} inches at 300 DPI')

IEEE style applied.
Figure size: 3.5 x 2.625 inches at 300 DPI


In [6]:
# Figure 1: Accuracy Degradation Curves
fig1, ax1 = plt.subplots(figsize=(IEEE_W, IEEE_H))

# Plot three lines
ax1.plot(x_rates, stgcn_means,
         color='#1f77b4', linestyle='--', marker='o',
         label='ST-GCN')

ax1.plot(x_rates, ctrgcn_means,
         color='#ff7f0e', linestyle='--', marker='s',
         label='CTR-GCN')

ax1.plot(x_rates, con_means,
         color='#2ca02c', linestyle='-', linewidth=1.8, marker='^',
         label='Consensus (ours)')

# Crossover vertical line
ax1.axvline(x=20, color='grey', linestyle=':', linewidth=0.9)
ax1.text(20.8, 4, 'Crossover\npoint',
         fontsize=7, color='grey', va='bottom')

# Axes
ax1.set_xlabel('Occlusion Rate (%)')
ax1.set_ylabel('Top-1 Accuracy (%)')
ax1.set_xlim(-2, 54)
ax1.set_ylim(0, 80)
ax1.set_xticks(x_rates)
ax1.set_xticklabels([f'{r}%' for r in x_rates])
ax1.yaxis.set_major_locator(ticker.MultipleLocator(10))

# Grid
ax1.grid(True, linewidth=0.4, alpha=0.6)

# Legend
ax1.legend(loc='upper right', framealpha=0.9, edgecolor='none')

# Remove top and right spines (IEEE style)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

plt.tight_layout()

out1 = FIGURES_DIR / 'accuracy_degradation_curves_70_10_20.png'
fig1.savefig(out1)
plt.show()
print(f'Saved: {out1}')

Saved: /content/drive/MyDrive/HRC_Research/figures/accuracy_degradation_curves_70_10_20.png


In [7]:
# Figure 2: Consensus Improvement Bars
fig2, ax2 = plt.subplots(figsize=(IEEE_W, IEEE_H))

x_labels = ['0%', '10%', '20%', '30%', '50%']
x_pos    = np.arange(len(x_labels))

bars = ax2.bar(x_pos, imp_values,
               color='#2ca02c', width=0.55,
               edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, imp_values):
    height = bar.get_height()
    if val >= 0:
        label = f'+{val:.2f} pp'
        y_pos = height + 0.08
    else:
        label = f'{val:.2f} pp'
        y_pos = height - 0.20
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        y_pos,
        label,
        ha='center', va='bottom', fontsize=7.5
    )

# Axes
ax2.set_xlabel('Occlusion Rate (%)')
ax2.set_ylabel('Improvement over Best Baseline (pp)')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_labels)

ax2.set_ylim(-1, 6)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(1))

# Y-axis grid only
ax2.grid(True, axis='y', linewidth=0.4, alpha=0.6)
ax2.set_axisbelow(True)

# Remove top and right spines (IEEE style)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()

out2 = FIGURES_DIR / 'consensus_improvement_bars_70_10_20.png'
fig2.savefig(out2)
plt.show()
print(f'Saved: {out2}')

Saved: /content/drive/MyDrive/HRC_Research/figures/consensus_improvement_bars_70_10_20.png


In [8]:
# Final confirmation
import os

figures = [
    FIGURES_DIR / 'accuracy_degradation_curves_70_10_20.png',
    FIGURES_DIR / 'consensus_improvement_bars_70_10_20.png',
]

print('Output verification:')
for f in figures:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f.name}: {size_kb:.1f} KB')
print('Done. Both figures saved to Drive.')

Output verification:
  accuracy_degradation_curves_70_10_20.png: 103.6 KB
  consensus_improvement_bars_70_10_20.png: 59.8 KB
Done. Both figures saved to Drive.
